# Silver Layer — PySpark + Delta Lake

Joins Bronze metadata + OCR Silver shards, normalises Turkish text, extracts province / ministry / party entities, writes Delta Silver tables.

Inputs:
- `data/bronze/yazili_soru_meta.parquet`
- `data/bronze/mp_index.parquet`
- `data/bronze/mp_to_guids.parquet`
- `data/silver/ocr_text/part-*.parquet` (synced from Drive)

Outputs (Delta tables under `data/silver/`):
- `yazili_soru_clean` — one row per önerge with merged önerge+cevap text + entities
- `mp_party` — clean MP→party→province lookup
- `cosign_edges` — placeholder for RQ3 (populated in 07_rq3)

In [ ]:
import sys, os
from pathlib import Path
sys.path.insert(0, str(Path('..').resolve() / 'src'))

from spark_utils import get_spark, BRONZE, SILVER, TABLES
spark = get_spark('silver-build', memory='6g')
spark.sparkContext.setLogLevel('WARN')

## 1. Load Bronze metadata + MP index

In [ ]:
meta = spark.read.parquet(str(BRONZE / 'yazili_soru_meta.parquet'))
mp_idx = spark.read.parquet(str(BRONZE / 'mp_index.parquet'))
mp_edges = spark.read.parquet(str(BRONZE / 'mp_to_guids.parquet'))

print('meta rows:', meta.count())
print('mp rows:', mp_idx.count())
print('mp→onerge edges:', mp_edges.count())
meta.printSchema()

## 2. Load OCR Silver shards

In [ ]:
ocr = spark.read.parquet(str(SILVER / 'ocr_text'))
print('OCR page rows:', ocr.count())
ocr.printSchema()

# Aggregate: text per (guid, doc_type) concatenated across pages
from pyspark.sql import functions as F
full = (
    ocr.orderBy('guid', 'doc_type', 'page')
       .groupBy('guid', 'doc_type')
       .agg(
           F.concat_ws('\n', F.collect_list('text')).alias('text'),
           F.avg('avg_conf').alias('avg_conf'),
           F.sum('n_lines').alias('n_lines'),
       )
)
full.show(5, truncate=80)

## 3. Pivot doc_type → wide schema (önerge_text, cevap_text)

In [ ]:
wide = (
    full.groupBy('guid')
        .pivot('doc_type', ['onerge', 'cevap'])
        .agg(
            F.first('text').alias('text'),
            F.first('avg_conf').alias('conf'),
        )
)
wide.show(3, truncate=60)

## 4. Turkish normalisation + entity extraction

In [ ]:
import unicodedata, re
from pyspark.sql.types import StringType, ArrayType

PROVINCES_81 = [
    'Adana','Adıyaman','Afyonkarahisar','Ağrı','Aksaray','Amasya','Ankara','Antalya','Ardahan','Artvin',
    'Aydın','Balıkesir','Bartın','Batman','Bayburt','Bilecik','Bingöl','Bitlis','Bolu','Burdur','Bursa',
    'Çanakkale','Çankırı','Çorum','Denizli','Diyarbakır','Düzce','Edirne','Elazığ','Erzincan','Erzurum',
    'Eskişehir','Gaziantep','Giresun','Gümüşhane','Hakkari','Hatay','Iğdır','Isparta','İstanbul','İzmir',
    'Kahramanmaraş','Karabük','Karaman','Kars','Kastamonu','Kayseri','Kırıkkale','Kırklareli','Kırşehir',
    'Kilis','Kocaeli','Konya','Kütahya','Malatya','Manisa','Mardin','Mersin','Muğla','Muş','Nevşehir',
    'Niğde','Ordu','Osmaniye','Rize','Sakarya','Samsun','Siirt','Sinop','Sivas','Şanlıurfa','Şırnak',
    'Tekirdağ','Tokat','Trabzon','Tunceli','Uşak','Van','Yalova','Yozgat','Zonguldak',
]
PROVINCES_SET = {p.upper() for p in PROVINCES_81}
PROV_PATTERN = re.compile(r'\b(' + '|'.join(map(re.escape, PROVINCES_81)) + r')\b', re.IGNORECASE)

def normalise(s):
    if not s: return ''
    s = unicodedata.normalize('NFKC', s)
    s = re.sub(r'\s+', ' ', s).strip()
    return s

def find_provinces(s):
    if not s: return []
    found = [m.group(1).title() for m in PROV_PATTERN.finditer(s)]
    # canonicalise typographic variants
    return sorted({p for p in found})

spark.udf.register('normalise', normalise, StringType())
spark.udf.register('find_provinces', find_provinces, ArrayType(StringType()))

## 5. Build yazili_soru_clean Silver table

In [ ]:
joined = (
    meta.alias('m')
        .join(wide.alias('w'), F.col('m.guid') == F.col('w.guid'), 'left')
        .drop(F.col('w.guid'))
)

silver = (
    joined
    .withColumn('ozet_norm', F.expr('normalise(özet)'))
    .withColumn('onerge_text', F.expr('normalise(onerge_text)'))
    .withColumn('cevap_text', F.expr('normalise(cevap_text)'))
    .withColumn('full_text', F.concat_ws('\n', 'ozet_norm', 'onerge_text', 'cevap_text'))
    .withColumn('provinces_mentioned', F.expr('find_provinces(full_text)'))
    .withColumn('year', F.split('geliş_tarihi', '\\.').getItem(2).cast('int'))
    .withColumn('month', F.split('geliş_tarihi', '\\.').getItem(1).cast('int'))
    .drop('onerge_pdf_url', 'cevap_pdf_url', 'detail_html_hash')
)
silver = silver.cache()
silver.show(3, truncate=60)
print('Silver rows:', silver.count())

## 6. Write Silver Delta tables

In [ ]:
from spark_utils import write_delta

write_delta(silver, TABLES['silver_yazili_soru_clean'], partition_by=['year'])
write_delta(mp_idx, TABLES['silver_mp_party'])
write_delta(mp_edges, TABLES['silver_cosign_edges'])

print('Silver tables written:')
for k in ['silver_yazili_soru_clean', 'silver_mp_party', 'silver_cosign_edges']:
    print(' ', k, '→', TABLES[k])

## 7. Smoke statistics

In [ ]:
silver.groupBy('year').count().orderBy('year').show()
silver.groupBy('muhatap_bakanlık').count().orderBy(F.desc('count')).show(15, truncate=60)
silver.select(F.size('provinces_mentioned').alias('n_prov')).describe().show()